In [1]:
import torch

class FCnet(torch.nn.Module):
    def __init__(self):
        super(FCnet, self).__init__()
        self.fc1 = torch.nn.Sequential(torch.nn.Linear(784,200), torch.nn.ReLU())
        self.fc2 = torch.nn.Sequential(torch.nn.Linear(200, 100), torch.nn.ReLU())
        self.fc3 = torch.nn.Sequential(torch.nn.Linear(100, 20), torch.nn.ReLU())
        self.fc4 = torch.nn.Linear(20,10)
    def forward(self, x):
        x1 = self.fc1(x)
        x2 = self.fc2(x1)
        x3 = self.fc3(x2)
        x4 = self.fc4(x3)
        return x4



In [2]:
import gzip, os, struct
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline


with gzip.open(os.path.join('..', 'temp', 'mnist', 'train-labels-idx1-ubyte.gz'), 'rb') as label:
    label = label.read()
    y_train_np = np.frombuffer(label, np.uint8, offset=8)
with gzip.open(os.path.join('..', 'temp', 'mnist', 'train-images-idx3-ubyte.gz'), 'rb') as img:
    img = img.read()
    x_train_np = np.frombuffer(img, np.uint8, offset=16).reshape(len(y_train_np), 28, 28)

with gzip.open(os.path.join('..', 'temp', 'mnist', 't10k-labels-idx1-ubyte.gz'), 'rb') as label:
    label = label.read()
    y_test_np = np.frombuffer(label, np.uint8, offset=8)
with gzip.open(os.path.join('..', 'temp', 'mnist', 't10k-images-idx3-ubyte.gz'), 'rb') as img:
    img = img.read()
    x_test_np = np.frombuffer(img, np.uint8, offset=16).reshape(len(y_test_np), 28, 28)

# print(os.getcwd())


# for i in range(9):
#     plt.subplot(3,3,i+1).set_title(y_train_np[i])
#     plt.imshow(x_train_np[i], 'gray')

# plt.tight_layout()
# plt.show()


In [6]:
from tqdm import tqdm
from torch.autograd import Variable
import time

# torch.backends.cudnn.benchmark = True
use_gpu = torch.cuda.is_available()

net = FCnet()
losser = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(net.parameters(),lr=0.001,weight_decay=0.0)

epochs = 100
batch_size = 1000
lr = 0.001

x_train = torch.from_numpy(x_train_np).float()
y_train = torch.from_numpy(y_train_np).long()
x_train, y_train = Variable(x_train), Variable(y_train)

if use_gpu:
    net = net.cuda()
    losser = losser.cuda()
    x_train = x_train.cuda()
    y_train = y_train.cuda()
start = time.time()
for epoch in tqdm(range(epochs)):
    index = 0
    for i in range(int(len(x_train)/batch_size)):
        batch_x = x_train[index:index+batch_size]
        batch_y = y_train[index:index+batch_size]
        prediction = net.forward(batch_x.view(batch_size, 784))
        loss = losser(prediction, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        index += batch_size

    print(loss)

print("spend: {}s".format(time.time()-start))
torch.save(net.state_dict(), os.path.join('..', 'temp', 'mnist', 'mnist.pth'))

2%|▏         | 2/100 [00:00<00:13,  7.32it/s]tensor(1.3634, device='cuda:0', grad_fn=<NllLossBackward>)
tensor(0.8570, device='cuda:0', grad_fn=<NllLossBackward>)
  4%|▍         | 4/100 [00:00<00:12,  7.63it/s]tensor(0.5792, device='cuda:0', grad_fn=<NllLossBackward>)
tensor(0.4482, device='cuda:0', grad_fn=<NllLossBackward>)
  6%|▌         | 6/100 [00:00<00:12,  7.69it/s]tensor(0.3749, device='cuda:0', grad_fn=<NllLossBackward>)
tensor(0.3289, device='cuda:0', grad_fn=<NllLossBackward>)
  8%|▊         | 8/100 [00:01<00:12,  7.51it/s]tensor(0.2971, device='cuda:0', grad_fn=<NllLossBackward>)
tensor(0.2724, device='cuda:0', grad_fn=<NllLossBackward>)
 10%|█         | 10/100 [00:01<00:11,  7.59it/s]tensor(0.2534, device='cuda:0', grad_fn=<NllLossBackward>)
tensor(0.2390, device='cuda:0', grad_fn=<NllLossBackward>)
 12%|█▏        | 12/100 [00:01<00:11,  7.52it/s]tensor(0.2280, device='cuda:0', grad_fn=<NllLossBackward>)
tensor(0.2188, device='cuda:0', grad_fn=<NllLossBackward>)
 14%|█▍   

In [6]:

x_test = torch.from_numpy(x_test_np).float()
y_test = torch.from_numpy(y_test_np).long()
x_test, y_test = Variable(x_test), Variable(y_test)

net.eval()
index = 0
result = torch.Tensor()
for i in range(int(x_test.shape[0]/batch_size)):
    test_batch_x = x_test[index:index+batch_size]
    label_pre = net.forward(test_batch_x.view(batch_size, 784))
    index += batch_size
    result = torch.cat((result, label_pre), 0)

loss = losser(result, y_test)
print(loss)




tensor(0.1242, grad_fn=<NllLossBackward>)
